<a href="https://colab.research.google.com/github/jyesing/HDB_resale_price/blob/main/2nd_Project_HDB_resale_since_2017.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd

df = pd.read_csv('/content/ResaleflatpricesbasedonregistrationdatefromJan2017onwards.csv')

df.head()

In [ ]:
df.describe()

In [ ]:
df.info()

In [ ]:
df.sort_values('month', ascending = False).head(10)

Mission 1: Find the top 5 towns with the highest average resale price in the most recent 12 months.

In [ ]:
df['month'] = pd.to_datetime(df['month'])

most_recent_month = pd.to_datetime('2025-12')

start_date_12_months = most_recent_month - pd.DateOffset(months=12)

df_recent_12_months = df[(df['month'] > start_date_12_months) & (df['month'] <= most_recent_month)].copy()

top_5_towns_recent_12_months = df_recent_12_months.groupby('town')['resale_price'].mean().sort_values(ascending = False).head(5)

top_5_towns_recent_12_months

Summary: Prices are highest in mature, centrally located towns due to limited supply and sustained demand, not just proximity to the city center.

Mission 2: Which flat_type has the highest price per sqm on average?

In [ ]:
df['psm'] = df['resale_price'] / df['floor_area_sqm']
df.groupby('flat_type')['psm'].mean().sort_values(ascending = False)

Summary: Smaller flats tend to have higher price per sqm due to a size effect, where fixed housing value is distributed over a smaller area. Buyers of smaller flats are more sensitive to absolute price rather than price efficiency, allowing price per sqm to be higher.

In [ ]:
import seaborn as sns
sns.scatterplot(data=df, x='floor_area_sqm', y='psm')

Mission 3: Does storey level affect price?

In [ ]:
temp_df_for_storey = df.copy()

temp_df_for_storey[['storey_min_str', 'storey_max_str']] = temp_df_for_storey['storey_range'].str.split(' TO ', expand=True)
temp_df_for_storey['storey_min'] = pd.to_numeric(temp_df_for_storey['storey_min_str'], errors='coerce')
temp_df_for_storey['storey_max'] = pd.to_numeric(temp_df_for_storey['storey_max_str'], errors='coerce')

temp_df_for_storey['storey_level'] = temp_df_for_storey['storey_max']

def categorize_storey(x):
    if pd.isna(x):
        return None
    elif x <= 6:
        return 'Low'
    elif x <= 12:
        return 'Mid'
    else:
        return 'High'

df['storey_category'] = temp_df_for_storey['storey_level'].apply(categorize_storey)

df.groupby('storey_category')['resale_price'].mean().sort_values(ascending=False)

The strong premium for high-floor units likely reflects both functional benefits and lifestyle preferences rather than purely structural differences, such as less noise, better view and more privacy.

Mission 4: Which towns have the fastest price growth from 2017 to latest?

In [ ]:
year = df['month'].dt.year
df['year'] = year

yearly_avg_price_by_town = df.pivot_table(values='resale_price', index='year', columns='town', aggfunc='mean')

price_latest_by_town = yearly_avg_price_by_town.loc[2024]
price_2017_by_town = yearly_avg_price_by_town.loc[2017]
town_growth = (price_latest_by_town - price_2017_by_town) / price_2017_by_town
town_growth.sort_values(ascending=False)

Summary: Higher growth rates in non-central towns are partly driven by a lower starting base, resulting in stronger percentage increases over time.

Mission 5: Does Flat age affects the price?

In [ ]:
df['flat_age'] = df['year'] - df['lease_commence_date']

df[['flat_age','resale_price']].corr()

In [ ]:
df['age_group'] = pd.cut(
    df['flat_age'],
    bins=[0,10,20,30,40,50,60,100],
    labels=['0-10','10-20','20-30','30-40','40-50','50-60','60+']
)

df.groupby('age_group')[['resale_price','floor_area_sqm']].mean()

Summary: Newer flats tend to have higher resale prices, but the relationship is not strictly linear. The relatively high prices of 20–30 year old flats are largely driven by their larger floor areas rather than age itself. This suggests that floor size is a stronger determinant of resale price than flat age in this range.

Mission 6: Convert remaining_lease into numeric years and compare with flat_age

In [ ]:
import re

def convert_lease(x):
  if not isinstance(x, str):
    return 0
  years_match = re.search(r'(\d+) years?', x)
  years = int(years_match.group(1)) if years_match else 0

  months_match = re.search(r'(\d+) months?', x)
  months = int(months_match.group(1)) if months_match else 0

  return years + months / 12

df['remaining_lease_years'] = df['remaining_lease'].apply(convert_lease)

df[['flat_age','remaining_lease_years']].corr()

In [ ]:
df['remaining_lease_groups'] = pd.cut(
    df['remaining_lease_years'],
    bins=[20,30,40,50,60,70,80,90, 100],
    labels=['20-30','30-40','40-50','50-60','60-70','70-80', '80-90', '90+']
)

df.groupby('remaining_lease_groups')[['resale_price','flat_age']].mean()

Sumamry: Remaining lease appears more linear because it better aligns with how value is perceived in the market, while flat age groups mix multiple confounding factors such as size and location.

Mission 7: Find outliers, then find which towns dominate these luxury outliers and what flat types are they?

In [ ]:
threshold = df['psm'].quantile(0.99)
outlier = df[df['psm'] > threshold]
outlier.head()

In [ ]:
outlier['flat_type'].value_counts().sort_values(ascending=False)

In [ ]:
outlier_pivot = outlier.pivot_table(values='psm', index='town', columns='flat_type', aggfunc='count', margins=True)
outlier_pivot_filled = outlier_pivot.fillna(0)
outlier_pivot_filled

Sumamry: Outliers in price per square meter are concentrated in 4-room flats and primarily located in mature, centrally located towns such as Toa Payoh, Queenstown, and Bukit Merah.

Mission 8: Find undervalued deals

In [ ]:
#compare similar flat type and remaining lease groups
expected = df_recent_12_months.groupby(['town','flat_type', 'remaining_lease_groups','storey_category'])['resale_price'].transform('median')

df_recent_12_months['price_gap']= df_recent_12_months['resale_price'] - expected
df_recent_12_months['price_gap_pct']= df_recent_12_months['price_gap'] / expected

undervalued = df_recent_12_months[df_recent_12_months['price_gap_pct'] <= -0.2]
undervalued['town'].value_counts()

In [ ]:
undervalued['flat_type'].value_counts()

In [ ]:
undervalued[['town','flat_type','resale_price','price_gap_pct']].sort_values('price_gap_pct').head(10)

Summary: Undervaluation is concentrated in mature towns like Toa Payoh and Kallang/Wjampoa, driven by high price dispersion within similar flat segments.

Mission 9: When did the price surge start?

In [ ]:
df.head()

In [ ]:
monthly = df['month'].dt.month
df['monthly'] = monthly


In [ ]:
monthly_avg_price = df.groupby(['year', 'monthly'])['resale_price'].median().reset_index()
monthly_avg_price['date'] = pd.to_datetime(monthly_avg_price['year'].astype(str) + '-' + monthly_avg_price['monthly'].astype(str))

import matplotlib.pyplot as plt

plt.figure(figsize=(15, 6))
sns.lineplot(data=monthly_avg_price, x='date', y='resale_price')
plt.title('Average Resale Price Over Time')
plt.xlabel('Date')
plt.ylabel('Average Resale Price')
plt.grid(True)
plt.show()

In [ ]:
baseline = monthly_avg_price[monthly_avg_price['year'] == 2019]['resale_price'].median()
peak = monthly_avg_price['resale_price'].max()
pct_increase = (peak - baseline) / baseline * 100
pct_increase

Summary: The monthly resale price trend shows a relatively stable period from 2017 to mid of 2018, followed by a slight dip during mid of 2018 to 2020. The price surge begins around mid-2020 and accelerates significantly after 2021. Comparing the pre-COVID baseline in 2019 to the peak period, resale prices increased by approximately 61.25%, indicating a strong post-pandemic structural shift in the HDB market.

Mission 10: Build a simple price prediction model

In [ ]:
features = ['floor_area_sqm','town','flat_type','storey_category', 'remaining_lease_groups']
df_model = pd.get_dummies(df[features + ['resale_price']], drop_first=True)
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression

X = df_model.drop('resale_price', axis=1)
y = df_model['resale_price']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = LinearRegression()
model.fit(X_train, y_train)

In [ ]:
def predict_price(town, flat_type, storey, floor_area_sqm, remaining_lease_group):
    new_flat = pd.DataFrame([
        {
            'town': town,
            'flat_type': flat_type,
            'storey_category': storey,
            'floor_area_sqm': floor_area_sqm,
            'remaining_lease_groups': remaining_lease_group,
        }
    ])

    new_flat = pd.get_dummies(new_flat)
    new_flat = new_flat.reindex(columns=X.columns, fill_value=0)

    return model.predict(new_flat)[0]

predict_price('PUNGGOL','4 ROOM','Mid', 95, '80-90')

Summary: Based on the trained model, a Punggol 4-room flat on a mid floor with 95 sqm and 80–90 years of remaining lease is estimated to have a resale value of approximately $500K, representing the expected market price given similar property characteristics.